<a href="https://colab.research.google.com/github/devesssi/DDPM-implementation-/blob/main/DDPM_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#imports
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import math

In [ ]:
#device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [ ]:
#variables
T = 1500
img_size = 28
batch_size = 128
epochs = 20
lr = 1e-3
base_channels = 64

In [ ]:
#dataset

transform_1 = transforms.Compose([transforms.ToTensor(),
                                transforms.Lambda(lambda x: x*2 - 1)])

train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform_1)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
#Forward diffusion parameters
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

In [ ]:
def forward_diffusion(x0, t):
  noise = torch.randn_like(x0)

  sqrt_ab = torch.sqrt(alpha_bars[t])[:, None, None, None]
  sqrt_one_minus_ab = torch.sqrt(1 - alpha_bars[t])[:, None, None, None]

  xt = sqrt_ab * x0 + sqrt_one_minus_ab * noise
  return xt, noise

In [ ]:
#Time embedding
class TimeEmbedding(nn.Module):
  def __init__(self, dim):
    super().__init__()
    self.dim = dim

  def forward(self, t):
    half = self.dim // 2
    freqs = torch.exp(-math.log(10000)*torch.arange(half, device = t.device)/(half-1))

    args = t[:,None] + freqs[None,:]
    output = torch.cat([torch.sin(args), torch.cos(args)], dim =1)
    return output

In [ ]:
#Residual blocks

class ResBlock(nn.Module):
  def __init__(self, in_ch, out_ch, time_dim):
    super().__init__()
    self.norm1 = nn.GroupNorm(32, in_ch)
    self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding = 1)

    self.norm2 = nn.GroupNorm(32, out_ch)
    self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding = 1)

    self.time_proj = nn.Linear(time_dim, out_ch)
    # self.skip = nn.Conv2d(in_ch, out_ch, 3, 1) if in_ch != out_ch else nn.Identity()

    self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

  def forward(self, x, t):
      h = self.conv1(torch.relu(self.norm1(x)))
      h = h + self.time_proj(t)[:, :, None, None]
      h = self.conv2(torch.relu(self.norm2(h)))

      return h + self.skip(x)

In [ ]:
#Attention block

class AttentionBlock(nn.Module):
  def __init__(self, ch):
    super().__init__()

    self.norm = nn.GroupNorm(32, ch)
    self.q = nn.Conv2d(ch, ch, 1)
    self.k = nn.Conv2d(ch, ch, 1)
    self.v = nn.Conv2d(ch, ch, 1)

    self.proj = nn.Conv2d(ch, ch, 1)

  def forward(self, x):
    B, C, H, W = x.shape
    h = self.norm(x)

    q = self.q(h).reshape(B, C, H*W)
    k = self.k(h).reshape(B, C, H*W)
    v = self.v(h).reshape(B, C, H*W)

    attn = torch.softmax(q.transpose(1,2)@k / math.sqrt(C), dim = -1)
    h = (attn @ v.transpose(1,2)).transpose(1,2).reshape(B, C, H, W)

    return x + self.proj(h)

In [ ]:
# DDPM UNet

class DDPMUNet(nn.Module):
  def __init__(self):
    super().__init__()
    time_dim = base_channels * 4

    self.time_mlp = nn.Sequential(
        TimeEmbedding(base_channels),
        nn.Linear(base_channels, time_dim),
        nn.ReLU(),
        nn.Linear(time_dim,time_dim)
    )

    self.conv_in = nn.Conv2d(1, base_channels, 3, padding = 1)

    #down

    self.down1 = ResBlock(base_channels, base_channels, time_dim)
    self.down2 = ResBlock(base_channels, base_channels * 2, time_dim)
    self.pool = nn.AvgPool2d(2)

    #bottleneck
    self.mid1 = ResBlock(base_channels * 2, base_channels * 2, time_dim)
    self.attn = AttentionBlock(base_channels * 2)
    self.mid2 = ResBlock(base_channels * 2, base_channels * 2, time_dim)

    #up
    self.up = nn.Upsample(scale_factor = 2, mode="nearest")
    self.up1 = ResBlock(base_channels * 2, base_channels, time_dim)
    self.up2 = ResBlock(base_channels * 2, base_channels, time_dim)

    self.conv_out = nn.Conv2d(base_channels, 1, 3, padding = 1)

  def forward(self, x, t):
    t = self.time_mlp(t.float())

    x = self.conv_in(x)
    d1 = self.down1(x, t)
    d2 = self.down2(d1, t)
    h = self.pool(d2)

    h = self.mid1(h, t)
    h = self.attn(h)
    h = self.mid2(h, t)

    h = self.up(h)
    h = self.up1(h, t)
    h = torch.cat([h, d1], dim = 1)
    h = self.up2(h, t)

    h = self.conv_out(h)

    return h

In [ ]:
model = DDPMUNet().to(device)
optimizer = optim.AdamW(model.parameters(), lr = lr)

In [ ]:
for epoch in range(epochs):
  for x, _ in train_loader:

    x = x.to(device)
    t = torch.randint(0, T, (x.shape[0],), device = device)
    xt, noise = forward_diffusion(x, t)

    noise_pred = model(xt, t)

    loss = ((noise_pred - noise)**2).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f"Epoch {epoch + 1}/{epochs} | Loss: {loss.item(): .4f}")